In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
import seaborn as sns
import sqlite3
import matplotlib as mpl
import matplotlib.pyplot as plt
import keras_tuner as kt

import copy
import random
import math
import sys
import gc

from sqlalchemy import create_engine
from tensorflow.keras import layers
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import CSVLogger
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import KFold,StratifiedKFold
from sklearn import model_selection
from keras_tuner.engine.tuner import Tuner

from datetime import datetime

In [ ]:
sys.path.insert(0, "C:/PlayerProjectionsHelper")

In [ ]:
from FeatureEngineering import FeatureEngineering
from FeatureResampling import FeatureResampling
from Normalization import Normalization

In [ ]:
dbConnectionString = "sqlite:///baseball_info.db"
engine = create_engine(dbConnectionString)

In [ ]:
timeSeriesHittingQuery = "SELECT SeasonStatsHitting.playerId,season,Bios.dob,plateAppearances,atBats,runs,hits,singles,doubles,triples,homeRuns,rbis,sacHits,sacFlies,hitByPitch,walks,intentionalWalks,strikeOuts,stolenBases,caughtStealing,groundedIntoDoublePlays,catcherInterference,battingAverage,onBasePercentage,sluggingPercentage,onBasePlusSlugging,runsPerTob,rbisPerBip,babip,spd,groundBallPercentage,flyBallPercentage,lineDrivePercentage,popUpPercentage,hardHitPercentage,barrelPercentage FROM SeasonStatsHitting INNER JOIN Bios on Bios.playerId = SeasonStatsHitting.playerId WHERE season BETWEEN 2015 and 2025 and season != 2020 ORDER BY SeasonStatsHitting.playerId,season"

df = pd.read_sql(timeSeriesHittingQuery, engine)

In [ ]:
FeatureEngineering.dropBattersWithNoPlateAppearances(df)
df = FeatureEngineering.dropPitchersWithOneSeasonOrLess(df)
FeatureEngineering.getAgeFromUnixTimeStamp(df)

In [ ]:
featureNum = len(df.columns)
windowSize = 4

windowedFeatures, windowedLabels = FeatureEngineering.getWindowedFeaturesAndLabelsForBatters(df, windowSize, featureNum)

In [ ]:
windowedFeaturesArray = np.array(windowedFeatures)
windowedLabelsArray = np.array(windowedLabels)

In [ ]:
binnedLabelsArray = FeatureResampling.getBinnedLabelsArrayForBatters            (windowedLabelsArray)
sortedBinCount    = FeatureResampling.getSortedBinCountFromBinnedArrayForBatters(binnedLabelsArray  )

In [ ]:
labels = list(sortedBinCount.keys())
count  = list(sortedBinCount.values())

plt.figure(figsize=(16, 10))
bar = plt.bar(labels, count)

_ = plt.xticks(labels, rotation="vertical")
_ = plt.bar_label(bar, count)

In [ ]:
binIndexes = FeatureResampling.getConcatenatedBinIndexesFromLabelsForBatters(labels)

In [ ]:
# stratifiedFeatures,stratifiedLabels = FeatureResampling.resampleFeaturesArrayForMinimumLabelsForBatters(windowedFeaturesArray, windowedLabelsArray, binnedLabelsArray, binIndexes, 11, 0, 20)

In [ ]:
# fullFeaturesArrayStratified = np.concatenate((windowedFeaturesArray, stratifiedFeatures), axis=0)
# fullLabelsArrayStratified   = np.concatenate((windowedLabelsArray  , stratifiedLabels  ), axis=0)

In [ ]:
# stratifiedBinnedLabelsArray = FeatureResampling.getBinnedLabelsArrayForBatters(fullLabelsArrayStratified)
# stratifiedSortedBinCount    = FeatureResampling.getSortedBinCountFromBinnedArrayForBatters(stratifiedBinnedLabelsArray)

In [ ]:
# stratifiedLabels = list(stratifiedSortedBinCount.keys())
# stratifiedCount  = list(stratifiedSortedBinCount.values())

# plt.figure(figsize=(8, 6))
# bar = plt.bar(stratifiedLabels, stratifiedCount)

# _ = plt.xticks(stratifiedLabels, rotation="vertical")
# _ = plt.bar_label(bar, stratifiedCount)

In [ ]:
# stratifiedTrainFeatures,stratifiedTestFeatures,stratifiedTrainLabels,stratifiedTestLabels = train_test_split(fullFeaturesArrayStratified, fullLabelsArrayStratified, stratify=stratifiedBinnedLabelsArray, test_size=0.1, random_state=42)

In [ ]:
# trainBinnedLabelsArrayS = FeatureResampling.getBinnedLabelsArrayForBatters(stratifiedTrainLabels)
# trainSortedBinCountS    = FeatureResampling.getSortedBinCountFromBinnedArrayForBatters(trainBinnedLabelsArrayS)

In [ ]:
# trainLabelsS = list(trainSortedBinCountS.keys())
# trainCountS  = list(trainSortedBinCountS.values())

# plt.figure(figsize=(12, 8))
# bar = plt.bar(trainLabelsS, trainCountS)

# _ = plt.xticks(trainLabelsS, rotation="vertical")
# _ = plt.bar_label(bar, trainCountS)

In [ ]:
# concatenatedBinIndexes = []

# for label in stratifiedBinnedLabelsArray:
#     concatenatedLabel = ""

#     for num in label:
#         concatenatedLabel = concatenatedLabel + str(num)

#     concatenatedBinIndexes.append(concatenatedLabel)

In [ ]:
tunerEarlyStop = EarlyStopping(monitor='val_loss', mode='min', patience=10, restore_best_weights=True)

In [ ]:
#l2/l1 regularization, dropout, activation function, optmizier, loss function, hidden units, hidden layers, batch size, batch normalization
def getModelForTuner(hp):
    hp_learning_rate     = hp.Float('learning_rate'    , min_value=1e-7, max_value=1e-3, sampling='log')
    hp_weight_decay      = hp.Float('weight_decay'     , min_value=1e-7, max_value=1e-3, sampling='log')   

    num_rnn_layers = hp.Int("num_rnn_layers", min_value=1, max_value=3)
    
    model = tf.keras.Sequential()
    model.add(layers.Masking(mask_value=0.0, input_shape=(3, 34)))

    for i in range(num_rnn_layers):
        hp_units             = hp.Int  (f'units_{i}'            , min_value=5, max_value=30, step=6)
        hp_dropout           = hp.Float(f'dropout_{i}'          , min_value=0.1, max_value=0.9, sampling='linear')     
        hp_recurrent_dropout = hp.Float(f'recurrent_dropout_{i}', min_value=0.1, max_value=0.9, sampling='linear')  
        hp_regularizer       = hp.Float(f'regularizer_{i}'      , min_value=1e-7, max_value=1e-3, sampling='log')  

        if i < num_rnn_layers - 1:
            model.add(layers.Bidirectional(layers.LSTM(units=hp_units, dropout=hp_dropout, recurrent_dropout=hp_recurrent_dropout, kernel_regularizer=regularizers.l2(hp_regularizer), return_sequences=True)))
        else:
            model.add(layers.Bidirectional(layers.LSTM(units=hp_units, dropout=hp_dropout, recurrent_dropout=hp_recurrent_dropout, kernel_regularizer=regularizers.l2(hp_regularizer), return_sequences=False)))
        
        model.add(layers.LayerNormalization())
        
        model.add(layers.Dense(units = 5))

    model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=hp_learning_rate, weight_decay=hp_weight_decay), loss=tf.keras.losses.Huber(), metrics=[tf.keras.metrics.R2Score(), "accuracy"])

    return model

In [ ]:
# tuner = kt.Hyperband(getModelForTuner,
#                      objective='loss',
#                      max_epochs=100,
#                      factor=3,
#                      directory='batter_tuner',
#                      project_name='batter_tuner_test_kfold')

In [ ]:
class KFoldTuner(Tuner):
    def run_trial(self, trial, x, y, *args, **kwargs):
        kFold = KFold(n_splits = 10, shuffle=True, random_state=42)

        epochs = trial.hyperparameters.get('tuner/epochs')

        batch_size = kwargs['batch_size']
        callbacks = kwargs['callbacks']

        valLossSum = 0.0

        for trainIndex,testIndex in kFold.split(x):
            tf.keras.backend.clear_session()
            tf.compat.v1.reset_default_graph()
            
            gc.collect()
            
            trainFeatures, valFeatures = x[trainIndex], x[testIndex]
            trainLabels  , valLabels   = y[trainIndex], y[testIndex]
    
            trainBinnedLabelsArrayT = FeatureResampling.getBinnedLabelsArrayForBatters(trainLabels)
            trainSortedBinCountT    = FeatureResampling.getSortedBinCountFromBinnedArrayForBatters(trainBinnedLabelsArrayT)

            trainLabelsBarT   = list(trainSortedBinCountT.keys())
            trainLabelsCountT = list(trainSortedBinCountT.values())

            trainBinIndexesT = FeatureResampling.getConcatenatedBinIndexesFromLabelsForBatters(trainLabelsBarT)
    
            maxCountPerCluster = trainLabelsCountT[0]

            stratifiedTrainFeaturesResampled,stratifiedTrainLabelsResampled = FeatureResampling.resampleFeaturesArrayForMinimumLabelsForBatters(trainFeatures, trainLabels, trainBinnedLabelsArrayT, trainBinIndexesT, 6, 0, 20)

            if len(stratifiedTrainFeaturesResampled) > 0:
                stratifiedTrainFeaturesResampled = np.concatenate((trainFeatures, stratifiedTrainFeaturesResampled), axis=0)
                stratifiedTrainLabelsResampled   = np.concatenate((trainLabels  , stratifiedTrainLabelsResampled  ), axis=0)
            else:
                stratifiedTrainFeaturesResampled = trainFeatures
                stratifiedTrainLabelsResampled = trainLabels
                
            interpolatedFeatures, interpolatedLabels = FeatureResampling.applySmoteToFeaturesAndLabelsForBatters(stratifiedTrainFeaturesResampled, stratifiedTrainLabelsResampled, 5, maxCountPerCluster, 0, 20)
        
            interpolatedFeatures = np.array(interpolatedFeatures)
            interpolatedFeatures = interpolatedFeatures.reshape(len(interpolatedFeatures), 3, 34)
            interpolatedLabels   = np.array(interpolatedLabels)
        
            concatenatedFeaturesArray = np.concatenate((stratifiedTrainFeaturesResampled , interpolatedFeatures), axis=0)
            concatenatedLabelsArray   = np.concatenate((stratifiedTrainLabelsResampled    , interpolatedLabels  ), axis=0)
        
            normalizedFeatures,normalizedLabels,featuresMean,featuresStd,labelsMean,labelsStd  = Normalization.normalizeTrainFeaturesAndLabelsForBatters(concatenatedFeaturesArray, concatenatedLabelsArray)
            normalizedValFeatures,normalizedValLabels                                          = Normalization.normalizeTestDataForBatters(valFeatures, valLabels, featuresMean, featuresStd, labelsMean, labelsStd)

            model = self.hypermodel.build(trial.hyperparameters)

            model.fit(normalizedFeatures, normalizedLabels, batch_size=batch_size, epochs=epochs, validation_data=(normalizedValFeatures, normalizedValLabels), callbacks=callbacks)

            valLoss = model.evaluate(normalizedValFeatures,normalizedValLabels, verbose=0)[0]

            valLossSum += valLoss

            tf.keras.backend.clear_session()
            tf.compat.v1.reset_default_graph()
            
            gc.collect()

        valLossMean = valLossSum / 10.0

        print(f"val_loss mean: {valLossMean}")

        self.oracle.update_trial(trial.trial_id, {'val_loss': valLossMean})

In [ ]:
# stratifiedFullFeaturesArray,stratifiedFullLabelsArray = FeatureResampling.resampleFeaturesArrayForMinimumLabelsForBatters(windowedFeaturesArray, windowedLabelsArray, trainBinnedLabelsArrayT, trainBinIndexesT, maxCountPerCluster, 0, 20)

In [ ]:
# normalizedTunerFeatures,normalizedTunerLabels,tunerFeaturesMean,tunerFeaturesStd,tunerLabelsMean,tunerLabelsStd = Normalization.normalizeTrainFeaturesAndLabelsForBatters(windowedFeaturesArray, windowedLabelsArray)

In [ ]:
tuner = KFoldTuner(  
                     hypermodel=getModelForTuner,
                     oracle=kt.oracles.HyperbandOracle(
                         objective='val_loss',
                         max_epochs=100,
                         factor=3,
                         seed=42
                     )
                  )

In [ ]:
# windowedFeaturesArray.shape

In [ ]:
# # bestModel = tuner.get_best_models(num_models=1)[0]

# best_hps=tuner.get_best_hyperparameters(num_trials=1)[0]

# print(best_hps.get('num_rnn_layers'))
# print(best_hps.get('learning_rate' ))
# print(best_hps.get('weight_decay'))
# print(best_hps.get('units_0'))
# print(best_hps.get('dropout_0'))
# print(best_hps.get('recurrent_dropout_0'))
# print(best_hps.get('regularizer_0'))

# print("-----------------")

# print(best_hps.get('units_1'))
# print(best_hps.get('dropout_1'))
# print(best_hps.get('recurrent_dropout_1'))
# print(best_hps.get('regularizer_1'))

# print("--------------------------")

# print(best_hps.get('units_2'))
# print(best_hps.get('dropout_2'))
# print(best_hps.get('recurrent_dropout_2'))
# print(best_hps.get('regularizer_2'))

In [ ]:
# def get_kfolds_bidirectional_model():    
#     model = tf.keras.Sequential([
#         layers.Masking(mask_value=0.0, input_shape=(2, 34)),
#         #normalizeInput,
#         layers.Bidirectional(layers.LSTM(units=29, dropout=0.39705292627585376, recurrent_dropout=0.7073788954336635, kernel_regularizer=regularizers.l2(1.0248324039665815e-06), return_sequences=False)),
#         layers.LayerNormalization(),
#         layers.Dense(units = 5)
#     ])

#     model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=0.0008126049044556143, weight_decay=0.00038223738778563054), loss=tf.keras.losses.Huber(), metrics=[tf.keras.metrics.R2Score(), "accuracy"])

#     return model

In [ ]:
bestModelSoFar = tuner.hypermodel.build(tuner.get_best_hyperparameters(num_trials=1)[0])

In [ ]:
bestModelSoFar.summary()

In [ ]:
fullBinIndexes = FeatureResampling.getConcatenatedBinIndexesFromLabelsForBatters(labels)

trainFeaturesResampled,trainLabelsResampled = FeatureResampling.resampleFeaturesArrayForMinimumLabelsForBatters(windowedFeaturesArray, windowedLabelsArray, binnedLabelsArray, fullBinIndexes, 6, 0, 20)

trainFeaturesResampled = np.concatenate((windowedFeaturesArray, trainFeaturesResampled), axis=0)
trainLabelsResampled   = np.concatenate((windowedLabelsArray  , trainLabelsResampled  ), axis=0)

fullFeaturesArraySmote,fullLabelsArraySmote = FeatureResampling.applySmoteToFeaturesAndLabelsForBatters(trainFeaturesResampled, trainLabelsResampled, 5, 703, 0, 20)

In [ ]:
trainFeaturesResampled.shape

In [ ]:
interpolatedFeatures = np.array(fullFeaturesArraySmote)
interpolatedFeatures = interpolatedFeatures.reshape(len(interpolatedFeatures), 3, 34)
interpolatedLabels   = np.array(fullLabelsArraySmote)

concatenatedFeaturesArray = np.concatenate((trainFeaturesResampled , interpolatedFeatures), axis=0)
concatenatedLabelsArray   = np.concatenate((trainLabelsResampled    , interpolatedLabels  ), axis=0)

In [ ]:
binnedLabelsArraySmote = FeatureResampling.getBinnedLabelsArrayForBatters            (concatenatedLabelsArray)
sortedBinCountSmote    = FeatureResampling.getSortedBinCountFromBinnedArrayForBatters(binnedLabelsArraySmote  )

In [ ]:
labelsSmote = list(sortedBinCountSmote.keys())
countSmote  = list(sortedBinCountSmote.values())

plt.figure(figsize=(16, 10))
bar = plt.bar(labelsSmote, countSmote)

_ = plt.xticks(labelsSmote, rotation="vertical")
_ = plt.bar_label(bar, countSmote)

In [ ]:
normalizedFullFeatures,normalizedFullLabels,featuresMean,featuresStd,labelsMean,labelsStd  = Normalization.normalizeTrainFeaturesAndLabelsForBatters(concatenatedFeaturesArray, concatenatedLabelsArray)

In [ ]:
fullEarlyStop = EarlyStopping(monitor='loss', mode='min', patience=20, restore_best_weights=True)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='loss', # Metric to monitor
    factor=0.2,         # Factor by which the learning rate will be reduced
    patience=10,        # Number of epochs with no improvement after which learning rate will be reduced
    min_lr=0.00001      # Lower bound on the learning rate
)
    
bestModelSoFar.fit(normalizedFullFeatures, normalizedFullLabels, batch_size=64, epochs=250, callbacks=[fullEarlyStop, reduce_lr])

In [ ]:
splits = 10

kFold = KFold(n_splits = splits, shuffle=True, random_state=42)

In [ ]:
featuresMean = None
featuresStd = None

labelsMean = None
labelsStd  = None

for trainIndex,testIndex in kFold.split(windowedFeaturesArray):
    trainFeatures, valFeatures = windowedFeaturesArray[trainIndex], windowedFeaturesArray[testIndex]
    trainLabels  , valLabels   = windowedLabelsArray  [trainIndex], windowedLabelsArray  [testIndex]
    
    trainBinnedLabelsArrayT = FeatureResampling.getBinnedLabelsArrayForBatters(trainLabels)
    trainSortedBinCountT    = FeatureResampling.getSortedBinCountFromBinnedArrayForBatters(trainBinnedLabelsArrayT)

    trainLabelsBarT   = list(trainSortedBinCountT.keys())
    trainLabelsCountT = list(trainSortedBinCountT.values())

    trainBinIndexesT = FeatureResampling.getConcatenatedBinIndexesFromLabelsForBatters(trainLabelsBarT)
    
    maxCountPerCluster = trainLabelsCountT[0]

    print(f"max clusters before undersampling: {maxCountPerCluster}")

    # trainFeatures,trainLabels = FeatureResampling.underSampleArraysForBatters(trainFeatures,trainLabels,trainBinnedLabelsArrayT,trainBinIndexesT, 200)

    # trainBinnedLabelsArrayT = FeatureResampling.getBinnedLabelsArrayForBatters(trainLabels)
    # trainSortedBinCountT    = FeatureResampling.getSortedBinCountFromBinnedArrayForBatters(trainBinnedLabelsArrayT)

    # trainLabelsBarT   = list(trainSortedBinCountT.keys())
    # trainLabelsCountT = list(trainSortedBinCountT.values())

    # trainBinIndexesT = FeatureResampling.getConcatenatedBinIndexesFromLabelsForBatters(trainLabelsBarT)
    
    # maxCountPerCluster = trainLabelsCountT[1]

    # print(f"max clusters after undersampling: {maxCountPerCluster}")

    stratifiedTrainFeaturesResampled,stratifiedTrainLabelsResampled = FeatureResampling.resampleFeaturesArrayForMinimumLabelsForBatters(trainFeatures, trainLabels, trainBinnedLabelsArrayT, trainBinIndexesT, 6, 0, 20)

    if len(stratifiedTrainFeaturesResampled) > 0:
        stratifiedTrainFeaturesResampled = np.concatenate((trainFeatures, stratifiedTrainFeaturesResampled), axis=0)
        stratifiedTrainLabelsResampled   = np.concatenate((trainLabels  , stratifiedTrainLabelsResampled  ), axis=0)
    else:
        stratifiedTrainFeaturesResampled = trainFeatures
        stratifiedTrainLabelsResampled = trainLabels
        
    interpolatedFeatures, interpolatedLabels = FeatureResampling.applySmoteToFeaturesAndLabelsForBatters(stratifiedTrainFeaturesResampled, stratifiedTrainLabelsResampled, 5, maxCountPerCluster, 0, 20)

    interpolatedFeatures = np.array(interpolatedFeatures)
    interpolatedFeatures = interpolatedFeatures.reshape(len(interpolatedFeatures), 3, 34)
    interpolatedLabels   = np.array(interpolatedLabels)

    concatenatedFeaturesArray = np.concatenate((stratifiedTrainFeaturesResampled , interpolatedFeatures), axis=0)
    concatenatedLabelsArray   = np.concatenate((stratifiedTrainLabelsResampled    , interpolatedLabels  ), axis=0)

    normalizedFeatures,normalizedLabels,featuresMean,featuresStd,labelsMean,labelsStd  = Normalization.normalizeTrainFeaturesAndLabelsForBatters(stratifiedTrainFeaturesResampled, stratifiedTrainLabelsResampled)
    normalizedValFeatures,normalizedValLabels                                          = Normalization.normalizeTestDataForBatters(valFeatures, valLabels, featuresMean, featuresStd, labelsMean, labelsStd)

    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', # Metric to monitor
    factor=0.2,         # Factor by which the learning rate will be reduced
    patience=10,        # Number of epochs with no improvement after which learning rate will be reduced
    min_lr=0.00001      # Lower bound on the learning rate
    )

    kFoldEarlyStop = EarlyStopping(monitor='val_loss', mode='min', patience=20, restore_best_weights=True)

    kFoldModel = get_kfolds_bidirectional_model()

    kFoldModel.fit(normalizedFeatures, normalizedLabels, batch_size=64, epochs=250, validation_data=(normalizedValFeatures, normalizedValLabels), callbacks=[kFoldEarlyStop, reduce_lr])

    # print("---------------------------------------------------------------------------")

    # print("---------------------------------------------------------------------------")

In [ ]:
baselineResults = kFoldModel.evaluate(normalizedTestFeatures, normalizedTestLabels)

In [ ]:
accuracy: 0.5699 - loss: 0.2038 - r2_score: 0.5511 

In [ ]:
for featureIndex in range(normalizedTestFeatures.shape[2]):
   shuffledTestFeatures = normalizedTestFeatures.copy()

   testColumnElements = [shuffledTestFeatures[i][j][featureIndex] for i in range(len(shuffledTestFeatures)) for j in range(len(shuffledTestFeatures[i]))]

   random.shuffle(testColumnElements)
   
   k = 0
   
   for i in range(len(shuffledTestFeatures)):
       for j in range(len(shuffledTestFeatures[i])):
           shuffledTestFeatures[i][j][featureIndex] = testColumnElements[k]
   
           k += 1

   shuffledResults = kFoldModel.evaluate(shuffledTestFeatures, normalizedTestLabels, verbose=0)

   ratio = shuffledResults[0] / baselineResults[0]

   print(f"{df.columns[featureIndex+2]}: {ratio}, {shuffledResults[0]}")

In [ ]:
trainFeatures[0]

In [ ]:
#763

season = 2025

hitterIds = []

hitterIdQuery = f"SELECT DISTINCT(PerGameStatsHitting.playerId),firstName,lastName from Games INNER JOIN PerGameStatsHitting ON Games.gameId = PerGameStatsHitting.gameId INNER JOIN Bios ON Bios.playerId = PerGameStatsHitting.playerId WHERE Games.season={season}"

connection = sqlite3.connect("C:/baseball_info.db")
cursor     = connection.cursor()

cursor.execute(hitterIdQuery)

rows = cursor.fetchall()

for row in rows:
    hitterIds.append((row[0],row[1],row[2]))

cursor    .close()
connection.close()

print(len(hitterIds))

In [ ]:
labelsMean

In [ ]:
indexToMean = {}

indexToMean[0] = labelsMean[0][0]
indexToMean[1] = labelsMean[0][1]
indexToMean[2] = labelsMean[0][2]
indexToMean[3] = labelsMean[0][3]
indexToMean[4] = labelsMean[0][4]

indexToStd = {}

indexToStd[0] = labelsStd[0][0]
indexToStd[1] = labelsStd[0][1]
indexToStd[2] = labelsStd[0][2]
indexToStd[3] = labelsStd[0][3]
indexToStd[4] = labelsStd[0][4]

In [ ]:
playerPredictions = []

for hitter in hitterIds:
    playerId = hitter[0]

    playerFeaturesQuery = f"SELECT Bios.playerId,season,Bios.dob,plateAppearances,atBats,runs,hits,singles,doubles,triples,homeRuns,rbis,sacHits,sacFlies,hitByPitch,walks,intentionalWalks,strikeOuts,stolenBases,caughtStealing,groundedIntoDoublePlays,catcherInterference,battingAverage,onBasePercentage,sluggingPercentage,onBasePlusSlugging,runsPerTob,rbisPerBip,babip,spd,groundBallPercentage,flyBallPercentage,lineDrivePercentage,popUpPercentage,hardHitPercentage,barrelPercentage FROM SeasonStatsHitting INNER JOIN Bios on Bios.playerId = SeasonStatsHitting.playerId WHERE season BETWEEN 2023 and 2025 and SeasonStatsHitting.playerId =\"{playerId}\" ORDER BY SeasonStatsHitting.playerId,season"

    playerDf = pd.read_sql(playerFeaturesQuery, engine)

    FeatureEngineering.getAgeFromUnixTimeStamp(playerDf)

    playerStats = []

    playerFrameArray = playerDf.values.tolist()
    
    for i in range(len(playerFrameArray)):
        playerStats.append(playerFrameArray[i][2:])
    
    if len(playerStats) < 3:
        diff = 3 - len(playerStats)
    
        while diff > 0:
            playerStats.append([0.0] * 34)
    
            diff -= 1

    fauxTrainLabels = [0,0,0,0,0]

    playerWindow = np.array([playerStats])
    fauxTrainLabelsArray = np.array([fauxTrainLabels])

    playerWindowNormalized,fauxTrainLabelsNormalized = Normalization.normalizeTestDataForBatters(playerWindow, fauxTrainLabelsArray, featuresMean, featuresStd, labelsMean, labelsStd)

    playerPrediction = bestModelSoFar.predict(playerWindowNormalized, verbose=0)

    normalizedPlayerPrediction = (playerId, hitter[1], hitter[2])

    for i in range(len(playerPrediction[0])):
        prediction = playerPrediction[0][i]
    
        curMean = indexToMean[i]
        curStd  = indexToStd [i]
    
        prediction = prediction * curStd + curMean
    
        if i < 4:
            prediction = max(0, math.ceil(prediction))
        else:
            prediction = max(0.0, round(prediction, 3))

        normalizedPlayerPrediction += (prediction,)

    playerPredictions.append(normalizedPlayerPrediction)

In [ ]:
maxRuns = 0.0
minRuns = 1000.0

maxHomeRuns = 0.0
minHomeRuns = 1000.0

maxRbis = 0.0
minRbis = 1000.0

maxStolenBases = 0.0
minStolenBases = 1000.0

maxOnBasePercentage = 0.0
minOnBasePercentage = 1.1

for player in playerPredictions:
    runs             = player[3]
    homeRuns         = player[4]
    rbis             = player[5]
    stolenBases      = player[6]
    onBasePercentage = player[7]

    if runs == -1:
        print(player)

    maxRuns = max(maxRuns, runs)
    minRuns = min(minRuns, runs)

    maxHomeRuns = max(maxHomeRuns, homeRuns)
    minHomeRuns = min(minHomeRuns, homeRuns)

    maxRbis = max(maxRbis, rbis)
    minRbis = min(minRbis, rbis)

    maxStolenBases = max(maxStolenBases, stolenBases)
    minStolenBases = min(minStolenBases, stolenBases)

    maxOnBasePercentage = max(maxOnBasePercentage, onBasePercentage)
    minOnBasePercentage = min(minOnBasePercentage, onBasePercentage)

print(f"{maxRuns}, {minRuns}")
print(f"{maxHomeRuns}, {minHomeRuns}")
print(f"{maxRbis}, {minRbis}")
print(f"{maxStolenBases}, {minStolenBases}")
print(f"{maxOnBasePercentage}, {minOnBasePercentage}")

In [ ]:
def normalize(num, maxValue, minValue):
    if maxValue == 0 and minValue == 0:
        return 0
    if maxValue == minValue:
        return 0
    
    normalizedValue = (num - minValue) / (maxValue - minValue)

    return normalizedValue

In [ ]:
for i in range(len(playerPredictions)):
    player = playerPredictions[i]
    
    runs             = player[3]
    homeRuns         = player[4]
    rbis             = player[5]
    stolenBases      = player[6]
    onBasePercentage = player[7]

    normalizedRuns        = normalize(runs, maxRuns, minRuns)
    normalizedHrs         = normalize(homeRuns, maxHomeRuns, minHomeRuns)
    normalizedObp         = normalize(onBasePercentage, maxOnBasePercentage, minOnBasePercentage)
    normalizedStolenBases = normalize(stolenBases, maxStolenBases, minStolenBases)
    normalizedRbis        = normalize(rbis, maxRbis, minRbis)

    overallGrade = normalizedRuns + normalizedHrs + normalizedObp + normalizedStolenBases + normalizedRbis

    player += (overallGrade,)

    playerPredictions[i] = player

In [ ]:
gradeSortedPredictions = sorted(playerPredictions, key=lambda player: player[8], reverse=True)

In [ ]:
for i in range(50):
    print(gradeSortedPredictions[i])

In [ ]:
predictionHeaders = ["id", "firstName", "lastName", "runs", "home runs", "rbis", "stolen bases", "obp", "grade"]

filePath = "./batting_predictions_2026.csv"

predictionsDf = pd.DataFrame(gradeSortedPredictions, columns=predictionHeaders)

predictionsDf.to_csv(filePath, index=False)

In [ ]:
bestModelSoFar.save("C:batterProjectionModels/best_2_18_2026/batter_projection_model.keras")

In [ ]:
import json

In [ ]:
np.savetxt('C:/best_2_18_2026/labelsStd.txt', labelsStd[0], fmt='%f', delimiter=',')

In [ ]:
featuresMean[0]